In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
target_urls = [
    "https://docs.cohere.com/docs/the-cohere-platform",
    "https://docs.cohere.com/docs/get-started-installation",
    "https://docs.cohere.com/docs/rerank",
    "https://docs.cohere.com/docs/cohere-embed"
]

In [4]:
# Initialized the loader to target only the <main> tag
loader = WebBaseLoader(
    web_paths=target_urls,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer("main") 
    )
)

# Scrape and load the content
docs = loader.load() # Here, the docs is a list, each object containing page_content and metadata

print(f"Successfully scraped {len(docs)} pages.")
print(f"Preview of the first page:\n{docs[0].page_content[:250]}...\n")

# Print the character count for each document
for doc in docs:
    print(f"{doc.metadata['source']} → {len(doc.page_content)} chars")

Successfully scraped 4 pages.
Preview of the first page:
Guides and conceptsAPI ReferenceRelease NotesLLMUCookbooksGet StartedIntroductionInstallationCreating a clientQuickstartPlaygroundFAQsModelsAn Overview of Cohere's ModelsCommandEmbedRerankAyaText GenerationIntroduction to Text Generation at CohereUsi...

https://docs.cohere.com/docs/the-cohere-platform → 6805 chars
https://docs.cohere.com/docs/get-started-installation → 2281 chars
https://docs.cohere.com/docs/rerank → 3222 chars
https://docs.cohere.com/docs/cohere-embed → 4447 chars


In [5]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="text-embedding-3-small", 
    chunk_size=256,         # Number of tokens
    chunk_overlap=32,  
)

splits = text_splitter.split_documents(docs)     # Split the documents

print(f"Split {len(docs)} documents into {len(splits)} chunks.")
print(f"Preview of the first chunk:\n{splits[0].page_content[:250]}...")

Split 4 documents into 29 chunks.
Preview of the first chunk:
Guides and conceptsAPI ReferenceRelease NotesLLMUCookbooksGet StartedIntroductionInstallationCreating a clientQuickstartPlaygroundFAQsModelsAn Overview of Cohere's ModelsCommandEmbedRerankAyaText GenerationIntroduction to Text Generation at CohereUsi...


In [6]:
# Storage and Indexing

from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# Initialize the embedding model
embeddings_model = OpenAIEmbeddings()

# Create the database and load the chunks
vectorstore = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings_model,
    location=":memory:",            # In RAM
    collection_name="cohere_docs"
)

# Retriever Interface
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Successfully embedded chunks and built the Qdrant vector store")

Successfully embedded chunks and built the Qdrant vector store


In [7]:
# Let's test the retriever with a relevant question
test_query = "What is the Rerank endpoint used for?"

# Fetch the most relevant chunks from our Qdrant database
test_results = retriever.invoke(test_query)

print(f"Retrieved {len(test_results)} chunks for the query: '{test_query}'\n")

# Print the content and the source URL of the top 2 results to verify
for i, doc in enumerate(test_results[:2]):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content Preview: {doc.page_content[:300]}...\n")

Retrieved 5 chunks for the query: 'What is the Rerank endpoint used for?'

--- Result 1 ---
Source: https://docs.cohere.com/docs/rerank
Content Preview: For each document included in a request, Rerank combines the tokens from the query with the tokens from the document and the combined total counts toward the context limit for a single document. If the combined number of tokens from the query and a given document exceeds the model’s context length f...

--- Result 2 ---
Source: https://docs.cohere.com/docs/rerank
Content Preview: Latest ModelDescriptionModalityEndpointsrerank-v4.0-proA multilingual model that allows for re-ranking English and non-english documents and semi-structured data (JSON). This is better suited for state-of-the-art quality and complex use-cases than its fast variant.TextRerankrerank-v4.0-fastA light v...



In [8]:
# Generation

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Prompt Template
template = """You are a helpful assistant. Answer the question based ONLY on the following context. 
If you don't know the answer from the context, just say that you don't know.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# A helper function to stitch our retrieved chunks into one big string of text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the final RAG Chain
# This connects: Retriever -> Formatter -> Prompt -> LLM -> Output Text
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Execute the chain
print(f"Asking the LLM: '{test_query}'...\n")
final_answer = rag_chain.invoke(test_query)

print("--- Final RAG Answer ---")
print(final_answer)

Asking the LLM: 'What is the Rerank endpoint used for?'...

--- Final RAG Answer ---
The Rerank endpoint is used to inject the intelligence of a language model into an existing search system. It sorts text inputs by semantic relevance to a specified query.


In [11]:
# A highly specific question based on the Rerank documentation
detailed_query = "What happens if the combined number of tokens from the query and a document exceeds the model's context length for Rerank?"

print(f"Question: {detailed_query}\n")

# See the embedding coordinates
query_vector = embeddings_model.embed_query(detailed_query)
print(f"Query Vector (First 5 of {len(query_vector)} dimensions):")
print(f"{query_vector[:5]}\n")

# Fetch the raw chunks and scores
raw_results = vectorstore.similarity_search_with_score(detailed_query, k=2)

print("Raw retrieved chunks from Qdrant")
for i, (doc, score) in enumerate(raw_results):
    print(f"\nChunk {i+1} | Similarity Score: {score:.4f} | Source: {doc.metadata['source']}")
    print(f"Content:\n{doc.page_content}")
    print("-" * 30)

# See exactly what LLM sees

# We strip the scores and metadata, and just join the text like our RAG chain does
context_string = format_docs([doc for doc, score in raw_results])
# We use the prompt template we built earlier to format the final string
exact_llm_input = prompt.format(context=context_string, question=detailed_query)

print("\nExact input sent to the LLM: \n")
print(exact_llm_input)
print("-" * 30)

# Final Answer
final_answer = llm.invoke(exact_llm_input)
print("\nLLM Output: \n")
print(final_answer.content)

Question: What happens if the combined number of tokens from the query and a document exceeds the model's context length for Rerank?

Query Vector (First 5 of 1536 dimensions):
[-0.01027099508792162, 0.011190052144229412, -0.007527853362262249, -0.003088664961978793, -0.006959581281989813]

Raw retrieved chunks from Qdrant

Chunk 1 | Similarity Score: 0.8923 | Source: https://docs.cohere.com/docs/rerank
Content:
For each document included in a request, Rerank combines the tokens from the query with the tokens from the document and the combined total counts toward the context limit for a single document. If the combined number of tokens from the query and a given document exceeds the model’s context length for a single document, the document will automatically get chunked and processed in multiple inferences. See our best practice guide for more info about formatting documents for the Rerank endpoint.Was this page helpful?YesNoEdit this pagePreviousAya Family of ModelsNextBuilt with
---